# V1-S10 — Novelty compute (semantic + structural CD-index + Kùzu graph)

This notebook **loads** the V1-S10 novelty artifacts and **spot-checks** the three novelty components. It does **not** recompute the heavy pipeline (semantic novelty + the 104 MB Kùzu graph already exist on disk) and runs **no network calls**.

Three lenses:

* **Semantic novelty** — per-paper embedding distance to prior same-field literature (`data/v1/novelty_semantic.parquet`). `sem_nov_mean` = distance to the prior-field centroid; `sem_nov_min` = distance to the nearest prior neighbour. `n_prior == 0` ⇒ undefined (NaN).
* **Structural CD-index** — the Funk–Owen-Smith disruption index (`scifield.novelty.cd_index`), validated **live** against the `cdindex` C extension (toy graph + a deterministic ~100-node graph; expect exact match / Pearson r ≈ 1.0).
* **Kùzu property graph** — the structural substrate (`data/v1/kuzu_graph`), queried read-only with Cypher COUNT queries per node label and relationship.

**Corpus-level CD** (`data/v1/cd_index.parquet`) is **GATED** on the OpenAlex `cited_by` forward-citation harvest (`data/v1/enrichment/cited_by.parquet`), which has not run yet. Section 6 documents the gate and degrades gracefully when the harvest is absent.

## 1. Setup / data load

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

# Repo-root sniff — notebook runs from notebooks/, code lives one dir up.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

DATA = repo_root / "data" / "v1"
SEMANTIC_PATH = DATA / "novelty_semantic.parquet"
TOPICS_PATH = DATA / "topics.parquet"
DUCKDB_PATH = DATA / "papers.duckdb"
KUZU_PATH = DATA / "kuzu_graph"
CITED_BY_PATH = DATA / "enrichment" / "cited_by.parquet"
CD_INDEX_PATH = DATA / "cd_index.parquet"

for p in (SEMANTIC_PATH, TOPICS_PATH, DUCKDB_PATH, KUZU_PATH):
    print(f"{'OK ' if p.exists() else 'MISSING'} {p}")
print(f"gated cited_by present : {CITED_BY_PATH.exists()}")
print(f"gated cd_index present : {CD_INDEX_PATH.exists()}")

OK  /Users/samersalman/Desktop/SciField/data/v1/novelty_semantic.parquet
OK  /Users/samersalman/Desktop/SciField/data/v1/topics.parquet
OK  /Users/samersalman/Desktop/SciField/data/v1/papers.duckdb
OK  /Users/samersalman/Desktop/SciField/data/v1/kuzu_graph
gated cited_by present : False
gated cd_index present : False


## 2. Semantic novelty

Load `data/v1/novelty_semantic.parquet` (one row per paper). Columns: `pmid, topic_id, year, n_prior, sem_nov_mean, sem_nov_min`. Papers with `n_prior == 0` are earliest-in-field and carry `NaN` for both scores — `describe()` ignores NaN.

In [2]:
sem = pd.read_parquet(SEMANTIC_PATH)
n_rows = len(sem)
n_no_prior = int((sem["n_prior"] == 0).sum())
print(f"rows                       : {n_rows:,}")
print(f"unique pmids               : {sem['pmid'].nunique():,}")
print(f"columns                    : {list(sem.columns)}")
print(f"n_prior == 0 (no prior)    : {n_no_prior:,}  ({n_no_prior / n_rows:.2%})")
print(f"sem_nov_mean NaN           : {int(sem['sem_nov_mean'].isna().sum()):,}")
print(f"sem_nov_min  NaN           : {int(sem['sem_nov_min'].isna().sum()):,}")

assert n_rows == 89_230, n_rows
print("\ndescribe() (NaN-ignored):")
display(sem[["sem_nov_mean", "sem_nov_min"]].describe())

rows                       : 89,230
unique pmids               : 89,230
columns                    : ['pmid', 'topic_id', 'year', 'n_prior', 'sem_nov_mean', 'sem_nov_min']
n_prior == 0 (no prior)    : 2,367  (2.65%)
sem_nov_mean NaN           : 2,367
sem_nov_min  NaN           : 2,372

describe() (NaN-ignored):


,sem_nov_mean,sem_nov_min
count,86863.000000,86858.000000
mean,0.493538,0.160743
std,0.169221,0.075267
min,0.059566,-0.000018
25%,0.363262,0.106898
50%,0.435641,0.146532
75%,0.644601,0.199765
max,0.997751,0.759402


## 3. Semantic ranking sanity (the plan's spot-check)

Join paper titles from `papers.duckdb` (`papers_distinct`) and eyeball the extremes. **Expectation:** review / consolidation papers — which restate prior work — should score **LOW** `sem_nov_mean`; first-in-field / methods papers — which depart from prior work — should score **HIGH**. This is a qualitative sanity table, not a metric.

We then repeat the within-one-topic check on a well-populated topic to confirm the ranking is sensible *inside* a field too.

In [3]:
# papers_distinct.pmid is VARCHAR; semantic pmid is int64. Cast on the SQL side
# so the join key aligns (a str-vs-int mismatch would silently produce no match).
con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
titles = con.execute("SELECT CAST(pmid AS BIGINT) AS pmid, title FROM papers_distinct").fetchdf()
con.close()
titles["pmid"] = titles["pmid"].astype(np.int64)

scored = sem.dropna(subset=["sem_nov_mean"]).merge(titles, on="pmid", how="left")
cov = scored["title"].notna().mean()
print(f"scored papers (sem_nov_mean defined): {len(scored):,}; title join coverage = {cov:.4f}")
assert cov > 0.95, f"title join coverage too low ({cov:.4f}) — pmid key mismatch?"


def _trim(s: str, n: int = 90) -> str:
    s = "" if s is None else str(s)
    return s if len(s) <= n else s[: n - 1] + "…"


show_cols = ["pmid", "topic_id", "year", "n_prior", "sem_nov_mean", "title"]
print("\nHIGHEST sem_nov_mean (expect first-in-field / methods / departures):")
hi = scored.sort_values("sem_nov_mean", ascending=False).head(6).copy()
hi["title"] = hi["title"].map(_trim)
display(hi[show_cols])

print("LOWEST sem_nov_mean (expect reviews / consolidations / restatements):")
lo = scored[scored["n_prior"] >= 5].sort_values("sem_nov_mean").head(6).copy()
lo["title"] = lo["title"].map(_trim)
display(lo[show_cols])

scored papers (sem_nov_mean defined): 86,863; title join coverage = 1.0000

HIGHEST sem_nov_mean (expect first-in-field / methods / departures):


,pmid,topic_id,year,n_prior,sem_nov_mean,title
44715,23001074,-1,2013,11829,0.997751,Murine gut microbiota and transcriptome are di...
21642,15621987,-1,2005,6266,0.994712,DNA array-based gene profiling: from surgical ...
60715,28902666,-1,2017,14701,0.994383,What is in a Pronoun?: Why Gender-fair Languag...
77304,36400581,-1,2023,18907,0.990937,Engineering functional 3-dimensional patient-d...
80149,37914572,-1,2024,19605,0.984191,Gastroenteropancreatic neuroendocrine carcinom...
69146,32370916,-1,2020,16782,0.976677,Tumor-specific near-infrared nanobody probe ra...


LOWEST sem_nov_mean (expect reviews / consolidations / restatements):


,pmid,topic_id,year,n_prior,sem_nov_mean,title
3491,9230864,53,1997,16,0.158710,Results of 1001 pancreatic resections for inva...
2531,9052434,145,1997,8,0.161738,Retrospective analysis of 70 operations for ga...
7901,10401733,53,1999,31,0.172154,Pancreatic cancer: a report of treatment and s...
43890,22717836,122,2012,8,0.174266,Effect of psychopathology on patient-perceived...
3810,9291403,53,1997,16,0.179051,Ductal adenocarcinoma of the body and tail of ...
4184,9361591,53,1997,16,0.186822,Low mortality following resection for pancreat...


In [4]:
# Within one well-populated topic: most- vs least-novel papers.
topic_sizes = scored["topic_id"].value_counts()
# Pick a sizeable non-noise topic (deterministic: largest topic_id >= 0).
candidate_topics = topic_sizes[topic_sizes.index >= 0]
focus_topic = int(candidate_topics.index[0])
tdf = scored[scored["topic_id"] == focus_topic]
print(f"focus topic_id={focus_topic}  (n papers scored={len(tdf):,})")

print("\nMost novel in topic:")
tm = tdf.sort_values("sem_nov_mean", ascending=False).head(4).copy()
tm["title"] = tm["title"].map(_trim)
display(tm[show_cols])

print("Least novel in topic:")
tl = tdf.sort_values("sem_nov_mean").head(4).copy()
tl["title"] = tl["title"].map(_trim)
display(tl[show_cols])

focus topic_id=0  (n papers scored=2,534)

Most novel in topic:


,pmid,topic_id,year,n_prior,sem_nov_mean,title
5362,9602824,0,1998,210,0.764511,Fatigue model to characterize cement-metal int...
14732,12011714,0,2002,489,0.666528,Computer-assisted fracture reduction of pelvic...
40264,21414744,0,2011,1394,0.664885,Factors affecting flexural strength in cement ...
52392,25754255,0,2015,1770,0.662009,The Effect of Taper Angle and Spline Geometry ...


Least novel in topic:


,pmid,topic_id,year,n_prior,sem_nov_mean,title
897,8698734,0,1996,73,0.262868,Revision of a failed cemented total hip prosth...
5807,9697999,0,1998,210,0.263963,A femoral component inserted without cement in...
1384,8816652,0,1996,73,0.264354,Revision of the acetabular component without c...
6502,9840632,0,1998,210,0.264990,Total hip arthroplasty with use of second-gene...


**Read:** if the high-`sem_nov_mean` rows skew toward primary / methodological / off-centroid work and the low-`sem_nov_mean` rows skew toward reviews, syntheses, and restatements of an established literature, the embedding-distance signal is behaving as designed. Same logic applies within the single focus topic above.

## 4. CD-index validation (live, fast, no network)

Validate the from-scratch CD index in `scifield.novelty.cd_index` against the installed `cdindex` C extension. This mirrors `tests/test_novelty_cd_index.py` but as an inline demonstration.

1. Reproduce the package's documented **toy graph** exactly (focal `4Z`, `t=5` ⇒ `1/6 ≈ 0.16667`) and assert our value equals the package's across several nodes/windows.
2. Build a deterministic ~100-node graph and report **Pearson r** (`np.corrcoef`) between our CD and `cdindex`'s CD across nodes (expect ≈ 1.0; gate ≥ 0.99).

In [5]:
import math
import random

import cdindex

from scifield.novelty.cd_index import CitationGraph, cd_index

# Toy graph from the cdindex docstring. Edge u->v means u cites v.
TOY_TIMES = {
    "0Z": 1992,
    "1Z": 1992,
    "2Z": 1993,
    "3Z": 1993,
    "4Z": 1995,
    "5Z": 1997,
    "6Z": 1998,
    "7Z": 1999,
    "8Z": 1999,
    "9Z": 1998,
    "10Z": 1997,
}
TOY_EDGES = [
    ("4Z", "2Z"),
    ("4Z", "0Z"),
    ("4Z", "1Z"),
    ("4Z", "3Z"),
    ("5Z", "2Z"),
    ("6Z", "2Z"),
    ("6Z", "4Z"),
    ("7Z", "4Z"),
    ("8Z", "4Z"),
    ("9Z", "4Z"),
    ("9Z", "1Z"),
    ("9Z", "3Z"),
    ("10Z", "4Z"),
]


def build_cdindex_graph(times, edges):
    g = cdindex.Graph()
    for name, t in times.items():
        g.add_vertex(name, int(t))
    seen = set()
    for u, v in edges:
        if (u, v) in seen or u == v:
            continue
        seen.add((u, v))
        g.add_edge(u, v)
    return g


ref = build_cdindex_graph(TOY_TIMES, TOY_EDGES)
ours = CitationGraph.from_edges(TOY_TIMES, TOY_EDGES)

pkg_4z = ref.cdindex("4Z", 5)
our_4z = cd_index(ours, "4Z", 5)
print(f"focal 4Z, t=5 : cdindex={pkg_4z:.6f}  ours={our_4z:.6f}  (documented 1/6={1/6:.6f})")
assert math.isclose(pkg_4z, 1.0 / 6.0, abs_tol=1e-12)
assert math.isclose(our_4z, pkg_4z, abs_tol=1e-9)

# Match across several focal nodes and windows.
mismatches = 0
for focal in TOY_TIMES:
    for t in (3, 5, 10):
        p = ref.cdindex(focal, t)
        m = cd_index(ours, focal, t)
        if p is None:
            if not math.isnan(m):
                mismatches += 1
        elif not math.isclose(m, p, abs_tol=1e-9):
            mismatches += 1
print(f"toy-graph cross-checks (11 nodes x 3 windows): {mismatches} mismatch(es)")
assert mismatches == 0
print("TOY GRAPH: our CD matches cdindex exactly.")

focal 4Z, t=5 : cdindex=0.166667  ours=0.166667  (documented 1/6=0.166667)
toy-graph cross-checks (11 nodes x 3 windows): 0 mismatch(es)
TOY GRAPH: our CD matches cdindex exactly.


In [6]:
# Deterministic ~120-node graph: newer nodes cite a handful of strictly-older nodes.
def make_random_graph(n=120, seed=1234):
    rng = random.Random(seed)
    names = [f"n{i}" for i in range(n)]
    times = {name: 2000 + rng.randint(0, 25) for name in names}
    edges, seen = [], set()
    for u in names:
        candidates = [v for v in names if times[v] < times[u]]
        rng.shuffle(candidates)
        for v in candidates[: rng.randint(0, 6)]:
            if (u, v) not in seen:
                seen.add((u, v))
                edges.append((u, v))
    return times, edges


times, edges = make_random_graph()
rref = build_cdindex_graph(times, edges)
rours = CitationGraph.from_edges(times, edges)

mine_vals, pkg_vals, exact_mismatch = [], [], 0
for focal in times:
    for t in (5, 10):
        p = rref.cdindex(focal, t)
        m = cd_index(rours, focal, t)
        if p is None:
            assert math.isnan(m), (focal, t, m)
            continue
        assert not math.isnan(m), f"ours NaN where cdindex defined: {focal} {t}"
        mine_vals.append(m)
        pkg_vals.append(p)
        if not math.isclose(m, p, abs_tol=1e-9):
            exact_mismatch += 1

r = float(np.corrcoef(np.array(mine_vals), np.array(pkg_vals))[0, 1])
print(f"non-degenerate focal nodes compared : {len(mine_vals)}")
print(f"exact mismatches vs cdindex          : {exact_mismatch}")
print(f"Pearson r (ours vs cdindex)          : {r:.6f}")
assert len(mine_vals) >= 30
assert exact_mismatch == 0
assert r >= 0.99, f"Pearson r below threshold: {r}"
print(f"\nCD-INDEX VALIDATION: PASS  (exact match; Pearson r = {r:.6f} >= 0.99)")

non-degenerate focal nodes compared : 209
exact mismatches vs cdindex          : 0
Pearson r (ours vs cdindex)          : 1.000000

CD-INDEX VALIDATION: PASS  (exact match; Pearson r = 1.000000 >= 0.99)


## 5. Kùzu property graph

Open `data/v1/kuzu_graph` **read-only** and run Cypher COUNT queries per node label (`Paper, Author, Journal, Institution, Topic`) and per relationship (`CITES, AUTHORED_BY, AFFILIATED_WITH, PUBLISHED_IN, ASSIGNED_TO`). The counts are asserted against the verified build values. A small sample query confirms the graph is queryable.

In [7]:
import kuzu

# Read-only open of the persistent single-file DB.
db = kuzu.Database(str(KUZU_PATH), read_only=True)
kcon = kuzu.Connection(db)


def count(pattern: str) -> int:
    res = kcon.execute(f"MATCH {pattern} RETURN count(*) AS n")
    return int(res.get_next()[0])


NODE_LABELS = ["Paper", "Author", "Institution", "Journal", "Topic"]
REL_LABELS = ["CITES", "AUTHORED_BY", "AFFILIATED_WITH", "PUBLISHED_IN", "ASSIGNED_TO"]

EXPECTED = {
    "Paper": 121_908,
    "Author": 238_119,
    "Institution": 29_159,
    "Journal": 10,
    "Topic": 149,
    "CITES": 574_478,
    "AUTHORED_BY": 622_294,
    "AFFILIATED_WITH": 505_815,
    "PUBLISHED_IN": 121_908,
    "ASSIGNED_TO": 67_821,
}

rows = []
for lbl in NODE_LABELS:
    n = count(f"(:{lbl})")
    rows.append(("node", lbl, n, EXPECTED[lbl], n == EXPECTED[lbl]))
for lbl in REL_LABELS:
    n = count(f"()-[:{lbl}]->()")
    rows.append(("rel", lbl, n, EXPECTED[lbl], n == EXPECTED[lbl]))

counts_df = pd.DataFrame(rows, columns=["kind", "label", "count", "expected", "match"])
display(counts_df)

all_match = bool(counts_df["match"].all())
print(f"\nall counts match expected: {all_match}")
assert all_match, counts_df[~counts_df["match"]]

,kind,label,count,expected,match
0,node,Paper,121908,121908,True
1,node,Author,238119,238119,True
2,node,Institution,29159,29159,True
3,node,Journal,10,10,True
4,node,Topic,149,149,True
5,rel,CITES,574478,574478,True
6,rel,AUTHORED_BY,622294,622294,True
7,rel,AFFILIATED_WITH,505815,505815,True
8,rel,PUBLISHED_IN,121908,121908,True
9,rel,ASSIGNED_TO,67821,67821,True



all counts match expected: True


In [8]:
# Tiny sample query: out-degree (CITES) of one paper, to show the graph is queryable.
sample = kcon.execute(
    """
    MATCH (p:Paper)-[:CITES]->(q:Paper)
    RETURN p.pmid AS pmid, count(*) AS out_degree
    ORDER BY out_degree DESC
    LIMIT 5
    """
).get_as_df()
print("Top-5 papers by corpus-internal CITES out-degree:")
display(sample)

top_pmid = str(sample.iloc[0]["pmid"])
one = kcon.execute(
    f"MATCH (p:Paper {{pmid: '{top_pmid}'}})-[:CITES]->() RETURN count(*) AS out_degree"
).get_next()[0]
print(f"\nspot-check: paper pmid={top_pmid} CITES out-degree = {int(one)}")

kcon.close()
db.close()

Top-5 papers by corpus-internal CITES out-degree:


,pmid,out_degree
0,33086352,257
1,35315607,256
2,21084588,230
3,30653039,179
4,16140830,171



spot-check: paper pmid=33086352 CITES out-degree = 257


## 6. Corpus CD-index (GATED on the OpenAlex cited_by harvest)

The corpus-level CD-index requires forward citations (who cites each focal work, and whether they also cite the focal work's references) — data the curated corpus does **not** contain. That is produced by the gated `cited_by` harvest:

```
scifield novelty harvest-citedby   # ~6.43M citers, ~4.4h, OpenAlex API — needs Samer's go
scifield novelty cd                # writes data/v1/cd_index.parquet
```

Until `data/v1/enrichment/cited_by.parquet` exists, the cell below prints the gate notice instead of computing.

**Documented approximation:** `compute_corpus_cd` derives the n_i / n_j split directly from each citer's `cites_focal_ref` flag and **omits the n_k term** (type-k citers — those that cite a reference of the focal work but not the focal work itself — are not present in the focal-citers-only harvest). With `denom = n_i + n_j` rather than `n_i + n_j + n_k`, the corpus CD is an **upward-biased approximation** of the true CD index. The live validation in Section 4 exercises the exact (n_k-inclusive) `cd_index`; the corpus driver trades exactness for harvest tractability.

In [9]:
cited_by = Path("data/v1/enrichment/cited_by.parquet")
if cited_by.exists():
    from scifield.novelty.cd_index import compute_corpus_cd

    cd = compute_corpus_cd(
        duckdb_path=Path("data/v1/papers.duckdb"),
        cited_by_parquet=cited_by,
        windows=(5, 10),
    )
    print(f"corpus CD computed: {len(cd):,} focal works")
    display(cd.describe())
else:
    print(
        "cited_by.parquet not present — corpus CD is gated on the OpenAlex harvest "
        "(~6.43M citers, ~4.4h). Run `scifield novelty harvest-citedby` after Samer's go, "
        "then `scifield novelty cd`."
    )

cited_by.parquet not present — corpus CD is gated on the OpenAlex harvest (~6.43M citers, ~4.4h). Run `scifield novelty harvest-citedby` after Samer's go, then `scifield novelty cd`.


## Summary

* **Semantic novelty** — 89,230 papers loaded; `n_prior == 0` count and `sem_nov_mean` / `sem_nov_min` distributions reported; high/low extremes and a within-topic ranking spot-checked against titles.
* **CD-index** — validated **live** against the `cdindex` C extension: toy graph exact (focal 4Z = 1/6), and Pearson r ≈ 1.0 (≥ 0.99) over a deterministic ~120-node graph with zero exact mismatches.
* **Kùzu graph** — opened read-only; all 5 node and 5 relationship counts asserted equal to the verified build values; a sample CITES out-degree query confirms queryability.
* **Corpus CD** — gated on the OpenAlex `cited_by` harvest; the guarded cell degrades to a notice. Corpus path omits n_k (focal-citers only) → documented upward-biased approximation.